# Physical AI最終課題：第4回実習 — UR5eをAI推論で動かす（MuJoCo）

**目標**：日本語指示 → 学習済みAIモデルの推論 → タスクID → UR5eの関節目標 → MuJoCoによるシミュレーション → MP4出力。

- 実行環境：Google Colab（CPUでも可。描画に時間がかかる場合あり）
- UR5e：Google DeepMind公式 MuJoCo Menagerie のモデルを使用
- AI：Hugging Faceの多言語Sentence Transformer（追加学習なし）
- **今回はグリッパーを装着せず、物体の把持や移送もしません。ROS 2は未接続です。** `LEFT_SAMPLE` 等は「左/右の作業に対応させた関節目標姿勢」の識別子です。
- ロボットの可動範囲・自他衝突・実機安全性の保証を目的とするものではありません。実機には接続しないでください。

### 操作
上から順にセルを実行します。最後にMP4とCSVをダウンロードします。最初は短いテスト動画、その後に提出要件の長さの動画を作成します。前回のColabとは**別のノートブック**として開いてください。

In [1]:
# セル1: 依存ライブラリのインストール（初回のみ時間がかかります）
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libosmesa6 libosmesa6-dev fonts-noto-cjk > /dev/null
%pip -q install "mujoco>=3.3,<4" "mujoco-menagerie==2026.10.3" "sentence-transformers>=3.0" "imageio[ffmpeg]" pillow pandas
print("必要なライブラリのインストールが完了しました。")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.6/30.6 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 8.0 MB/s eta 0:00:00
必要なライブラリのインストールが完了しました。


## 1. AI推論モデルと安全ゲート

文章埋め込みで登録済み作業とのコサイン類似度を計算し、1位を選びます。誤指令防止の**簡易確認**として、1位の類似度と2位との差が小さければロボットへ送らず停止します。これは実機安全基準ではありません。今回は確実に検証できる明確な指示から始めます。

In [2]:
# セル2: 学習済みAIモデルを準備
import os
os.environ["MUJOCO_GL"] = "osmesa"  # mujoco を import するより前に設定
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
text_model = SentenceTransformer(MODEL_ID, device="cpu")

# 作業IDと学習済みモデルへ比較させる日本語説明
TASK_TEXTS = {
    "LEFT_SAMPLE": "左の試料を測定場所へ移動して",
    "RIGHT_SAMPLE": "右の試料を測定場所へ移動して",
    "HOME": "アームを待機位置に戻して",
}
TASK_NAMES = list(TASK_TEXTS)
reference_embeddings = text_model.encode(
    [TASK_TEXTS[k] for k in TASK_NAMES], normalize_embeddings=True
)
print("AIモデル読み込み完了:", MODEL_ID)
print("登録タスク:", TASK_NAMES)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

AIモデル読み込み完了: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
登録タスク: ['LEFT_SAMPLE', 'RIGHT_SAMPLE', 'HOME']


In [3]:
# セル3: 推論と簡易的な棄却条件
MIN_SCORE = 0.60
MIN_MARGIN = 0.04

def infer_task(instruction):
    start = time.perf_counter()
    vector = text_model.encode([instruction], normalize_embeddings=True)[0]
    values = reference_embeddings @ vector
    order = np.argsort(values)[::-1]
    best, second = int(order[0]), int(order[1])
    chosen = TASK_NAMES[best]
    score, margin = float(values[best]), float(values[best] - values[second])
    # 語の矛盾だけを検知する確認用チェック（この規則自体はAIではない）
    contradictory = ("右" in instruction and "左" in instruction)
    if chosen == "LEFT_SAMPLE" and ("右" in instruction or "試料B" in instruction):
        contradictory = True
    if chosen == "RIGHT_SAMPLE" and ("左" in instruction or "試料A" in instruction):
        contradictory = True
    accepted = score >= MIN_SCORE and margin >= MIN_MARGIN and not contradictory
    return {
        "instruction": instruction,
        "proposed_task": chosen,
        "task_id": chosen if accepted else "REJECT",
        "score": round(score, 4),
        "margin": round(margin, 4),
        "inference_seconds": round(time.perf_counter() - start, 4),
        "all_scores": {TASK_NAMES[i]: round(float(values[i]), 4) for i in range(len(TASK_NAMES))},
    }

TEST_INPUTS = [
    "左の試料を測定場所へ移動して",
    "右の試料を測定場所へ移動して",
    "アームを待機位置に戻して",
    "右と左のどちらかの試料を運んで",   # 曖昧な指示を棄却できるか確認
]
for s in TEST_INPUTS:
    d = infer_task(s)
    print(f"入力: {s} → {d['task_id']} | score={d['score']:.4f} margin={d['margin']:.4f}")
assert infer_task(TEST_INPUTS[0])["task_id"] == "LEFT_SAMPLE", "左の指示が棄却・誤判定されました。結果を確認してください。"
assert infer_task(TEST_INPUTS[1])["task_id"] == "RIGHT_SAMPLE", "右の指示が棄却・誤判定されました。結果を確認してください。"
assert infer_task(TEST_INPUTS[2])["task_id"] == "HOME", "待機指示が棄却・誤判定されました。結果を確認してください。"
print("明確な3指示の動作確認に成功しました。")

入力: 左の試料を測定場所へ移動して → LEFT_SAMPLE | score=1.0000 margin=0.0597
入力: 右の試料を測定場所へ移動して → RIGHT_SAMPLE | score=1.0000 margin=0.0597
入力: アームを待機位置に戻して → HOME | score=1.0000 margin=0.5795
入力: 右と左のどちらかの試料を運んで → REJECT | score=0.7515 margin=0.0183
明確な3指示の動作確認に成功しました。


## 2. UR5e公式モデルをMuJoCoで読み込む

外部モデルは実行時にMenagerie公式パッケージから取得します。元モデルのライセンスはBSD-3-Clauseです。`home`姿勢を使用し、目標関節角はそこから一部を変化させます。**試料の実際の位置は測定していません。**

In [4]:
# セル4: UR5e のモデルをダウンロードして読み込み
import mujoco
import mujoco_menagerie as mm

robot_model = mm.load("universal_robots_ur5e")  # 初回ネット接続が必要
robot_data = mujoco.MjData(robot_model)

home_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_KEY, "home")
assert home_id >= 0, "UR5eのhomeキーフレームがありません。"
mujoco.mj_resetDataKeyframe(robot_model, robot_data, home_id)
mujoco.mj_forward(robot_model, robot_data)

JOINT_NAMES = [
    "shoulder_pan_joint", "shoulder_lift_joint", "elbow_joint",
    "wrist_1_joint", "wrist_2_joint", "wrist_3_joint"
]
assert robot_model.nq == 6 and robot_model.nu == 6, (robot_model.nq, robot_model.nu)
for idx, name in enumerate(JOINT_NAMES):
    joint_id = mujoco.mj_name2id(robot_model, mujoco.mjtObj.mjOBJ_JOINT, name)
    assert joint_id >= 0 and int(robot_model.jnt_qposadr[joint_id]) == idx, name

q_home = robot_data.qpos.copy()
q_left = q_home.copy()
q_left[0] = -2.25  # pan を左側の目標角へ（デモ用の関節角）
q_left[1] = -1.30
q_right = q_home.copy()
q_right[0] = -0.90
q_right[1] = -1.30

TARGETS = {"LEFT_SAMPLE": q_left, "RIGHT_SAMPLE": q_right, "HOME": q_home}
for name, target in TARGETS.items():
    assert np.all(np.isfinite(target)), name
    assert np.all(target >= robot_model.actuator_ctrlrange[:, 0])
    assert np.all(target <= robot_model.actuator_ctrlrange[:, 1])
    print(name, "目標関節角[rad]:", np.round(target, 3))
print("UR5eモデル確認:", robot_model.nq, "関節,", robot_model.nu, "アクチュエータ")

LEFT_SAMPLE 目標関節角[rad]: [-2.25  -1.3    1.571 -1.571 -1.571  0.   ]
RIGHT_SAMPLE 目標関節角[rad]: [-0.9   -1.3    1.571 -1.571 -1.571  0.   ]
HOME 目標関節角[rad]: [-1.571 -1.571  1.571 -1.571 -1.571  0.   ]
UR5eモデル確認: 6 関節, 6 アクチュエータ


## 3. 動画を記録する

左／右のタスクでは、推論したタスクIDに対応する**事前設定の関節角**を位置制御で追従します。MoveIt 2や逆運動学による障害物回避・経路計画はまだありません。映像上には入力文、AIの推論結果、シミュレーション状態を重ねて表示します。

In [21]:
# セル5: 動画生成関数
import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Video, display
from pathlib import Path
import os
import numpy as np
import pandas as pd
import mujoco

VIDEO_FPS = 8   # CPU描画の負荷を考えた設定（8 fps）
WIDTH, HEIGHT = 640, 480
FONT_PATH = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
FONT = ImageFont.truetype(FONT_PATH, 19) if os.path.exists(FONT_PATH) else ImageFont.load_default()
SMALL_FONT = ImageFont.truetype(FONT_PATH, 14) if os.path.exists(FONT_PATH) else ImageFont.load_default()


def add_overlay(image, main_text, sub_text, state_text):
    picture = Image.fromarray(image).convert("RGB")
    draw = ImageDraw.Draw(picture)
    draw.rectangle((0, 0, WIDTH, 101), fill=(13, 26, 40))
    draw.text((14, 10), main_text, font=FONT, fill=(235, 245, 255))
    draw.text((14, 42), sub_text, font=SMALL_FONT, fill=(185, 230, 240))
    draw.text((14, 68), state_text, font=SMALL_FONT, fill=(255, 215, 145))
    draw.rectangle((0, HEIGHT - 32, WIDTH, HEIGHT), fill=(13, 26, 40))
    draw.text((14, HEIGHT - 27), "MuJoCo UR5e | AI selects task; fixed joint targets | NO GRIPPER", font=SMALL_FONT, fill="white")
    return np.asarray(picture)


def record_demo(output_path, full=False):
    # 一度モデルをhomeに戻し、毎回同じ初期条件で実行
    d = mujoco.MjData(robot_model)
    mujoco.mj_resetDataKeyframe(robot_model, d, home_id)
    mujoco.mj_forward(robot_model, d)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat[:] = [0.0, 0.0, 0.36]
    cam.distance = 1.8
    cam.azimuth = 125
    cam.elevation = -23

    # 生成する実行指示。毎回infer_taskでAI推論してから実行する
    instructions = (
        [
            "左側の部品を検査台の前へ動かして",
            "最初の姿勢へ戻って",
            "右にある試料を計測場所へ移動して",
            "右と左のどちらかの試料を運んで",
            "最初の姿勢へ戻って",
            "左側の部品を検査台の前へ動かして",
        ]
        if full else
        [
            "左の試料を測定場所へ移動して",
            "右の試料を測定場所へ移動して",
            "アームを待機位置に戻して",
        ]
    )
    segment_seconds = 9 if full else 4
    intro_seconds = 5 if full else 2
    outro_seconds = 6 if full else 2
    move_seconds = segment_seconds - 1.5
    steps_per_frame = max(1, int(round((1.0 / VIDEO_FPS) / robot_model.opt.timestep)))
    records = []
    output_path = str(output_path)

    with mujoco.Renderer(robot_model, height=HEIGHT, width=WIDTH) as renderer:
        with imageio.get_writer(output_path, fps=VIDEO_FPS, codec="libx264", quality=7, macro_block_size=16) as writer:
            def emit(text, detail, status):
                renderer.update_scene(d, camera=cam)
                pixels = renderer.render().copy()
                writer.append_data(add_overlay(pixels, text, detail, status))

            for _ in range(intro_seconds * VIDEO_FPS):
                d.ctrl[:] = q_home
                mujoco.mj_step(robot_model, d, nstep=steps_per_frame)
                emit("Physical AI: UR5e task selection", "日本語の指示 → 学習済みAI推論 → UR5e", "モデル：UR5eのみ / 物体把持なし")

            for instruction in instructions:
                result = infer_task(instruction)  # モデルの推論結果をロボット制御へ使う
                selected = result["task_id"]
                start_q = d.qpos.copy()
                target_q = TARGETS.get(selected, start_q)
                print(f"AI推論: {instruction} => {selected} (類似度={result['score']:.3f}, 差={result['margin']:.3f})")
                for frame_i in range(segment_seconds * VIDEO_FPS):
                    t = min((frame_i + 1) / (move_seconds * VIDEO_FPS), 1.0)
                    ease = t*t*(3.0-2.0*t)
                    d.ctrl[:] = start_q + (target_q - start_q)*ease
                    mujoco.mj_step(robot_model, d, nstep=steps_per_frame)
                    emit("指示: " + instruction,
                         f"AI task = {selected}  |  similarity={result['score']:.3f}  margin={result['margin']:.3f}",
                         "推論に基づく目標関節角へ移動" if selected != "REJECT" else "判定保留：動作しない")
                err = float(np.max(np.abs(d.qpos - target_q)))
                records.append({**{k:v for k,v in result.items() if k != "all_scores"},
                                "max_joint_error_rad": round(err, 4),
                                "target_reached_within_0p12rad": bool(err < 0.12) if selected != "REJECT" else None})

            for _ in range(outro_seconds * VIDEO_FPS):
                d.ctrl[:] = q_home
                mujoco.mj_step(robot_model, d, nstep=steps_per_frame)
                emit("終了：AI推論によりタスクを選択", "出力動画と実行ログを保存しました", "把持・搬送・ROS 2は次段階の課題")

    result_csv = output_path.replace(".mp4", "_log.csv")
    pd.DataFrame(records).to_csv(result_csv, index=False, encoding="utf-8-sig")
    duration = intro_seconds + segment_seconds * len(instructions) + outro_seconds
    print(f"動画保存: {output_path}  長さ={duration} 秒")
    print(f"結果ログ: {result_csv}")
    return output_path, result_csv

## 4. 最初に短い動画を作り、動作を確認

以下のセルが成功すると、約16秒のテストMP4を表示します。提出用の長さ（1～3分）には達していませんが、動作と描画を確認するためのものです。**ここで結果の画面をChatGPTに共有すると、以降の問題を切り分けやすくなります。**

In [22]:
# セル6: まず短いテスト動画（約16秒）
short_mp4, short_csv = record_demo("ur5e_ai_short_test.mp4", full=False)
display(Video(short_mp4, embed=True, width=640))
print(pd.read_csv(short_csv).to_string(index=False))

AI推論: 左の試料を測定場所へ移動して => LEFT_SAMPLE (類似度=1.000, 差=0.060)
AI推論: 右の試料を測定場所へ移動して => RIGHT_SAMPLE (類似度=1.000, 差=0.060)
AI推論: アームを待機位置に戻して => HOME (類似度=1.000, 差=0.580)
動画保存: ur5e_ai_short_test.mp4  長さ=16 秒
結果ログ: ur5e_ai_short_test_log.csv


   instruction proposed_task      task_id  score  margin  inference_seconds  max_joint_error_rad  target_reached_within_0p12rad
左の試料を測定場所へ移動して   LEFT_SAMPLE  LEFT_SAMPLE    1.0  0.0597             0.0908               0.0129                           True
右の試料を測定場所へ移動して  RIGHT_SAMPLE RIGHT_SAMPLE    1.0  0.0597             0.0387               0.0129                           True
  アームを待機位置に戻して          HOME         HOME    1.0  0.5795             0.0452               0.0081                           True


In [7]:

from google.colab import files

files.download("ur5e_ai_short_test.mp4")
files.download("ur5e_ai_short_test_log.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:

# 追加演習：未知の日本語指示10件でAIの性能を評価

import pandas as pd

test_cases = [
    ("左側の部品を検査台の前へ動かして", "LEFT_SAMPLE"),
    ("試料Aを測定エリアへ近づけて", "LEFT_SAMPLE"),
    ("右側の部品を検査台まで運んで", "RIGHT_SAMPLE"),
    ("右にある試料を計測場所へ移動して", "RIGHT_SAMPLE"),
    ("試料Bを測定台に持っていって", "RIGHT_SAMPLE"),
    ("最初の姿勢へ戻って", "HOME"),
    ("ロボットアームをホーム位置にして", "HOME"),
    ("右と左のどちらかの試料を運んで", "REJECT"),
    ("適当な部品を動かして", "REJECT"),
    ("どの試料でもいいので移動して", "REJECT"),
]

results = []

for instruction, expected in test_cases:
    result = infer_task(instruction)
    predicted = result["task_id"]

    results.append({
        "instruction": instruction,
        "expected": expected,
        "predicted": predicted,
        "correct": predicted == expected,
        "score": result["score"],
        "margin": result["margin"],
    })

df_test = pd.DataFrame(results)

print(df_test.to_string(index=False))
print()
print("正答率:",
      round(df_test["correct"].mean() * 100, 1), "%")

df_test.to_csv(
    "ur5e_ai_instruction_test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV保存完了")


     instruction     expected    predicted  correct  score  margin
左側の部品を検査台の前へ動かして  LEFT_SAMPLE  LEFT_SAMPLE     True 0.7765  0.1073
  試料Aを測定エリアへ近づけて  LEFT_SAMPLE       REJECT    False 0.8369  0.0319
  右側の部品を検査台まで運んで RIGHT_SAMPLE       REJECT    False 0.7312  0.0216
右にある試料を計測場所へ移動して RIGHT_SAMPLE RIGHT_SAMPLE     True 0.9876  0.0522
  試料Bを測定台に持っていって RIGHT_SAMPLE       REJECT    False 0.7540  0.0336
       最初の姿勢へ戻って         HOME         HOME     True 0.7386  0.3480
ロボットアームをホーム位置にして         HOME       REJECT    False 0.5430  0.2501
 右と左のどちらかの試料を運んで       REJECT       REJECT     True 0.7515  0.0183
      適当な部品を動かして       REJECT       REJECT     True 0.5162  0.0360
  どの試料でもいいので移動して       REJECT       REJECT     True 0.5841  0.0200

正答率: 60.0 %
CSV保存完了


In [9]:
from google.colab import files

files.download("ur5e_ai_instruction_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. 提出条件の長さを満たすデモ動画

テスト動画でロボットが左右に動くことを確認できた場合だけ、次を実行してください。約65秒のMP4を出力します（提出条件の「1～3分」に入る長さ）。**この動画単体で最終課題の提出準備が完成するわけではありません。** 対象物の把持なし・ROS 2未使用・モデル推論とロボット動作の説明はレポートに明記してください。

In [23]:
# セル7: 提出候補の長さ（65秒）、描画に数分かかる場合あり
final_mp4, final_csv = record_demo("ur5e_ai_demo_65s_v2.mp4", full=True)
display(Video(final_mp4, embed=True, width=640))
print(pd.read_csv(final_csv)[["instruction", "task_id", "score", "margin", "max_joint_error_rad", "target_reached_within_0p12rad"]].to_string(index=False))

AI推論: 左側の部品を検査台の前へ動かして => LEFT_SAMPLE (類似度=0.776, 差=0.107)
AI推論: 最初の姿勢へ戻って => HOME (類似度=0.739, 差=0.348)
AI推論: 右にある試料を計測場所へ移動して => RIGHT_SAMPLE (類似度=0.988, 差=0.052)
AI推論: 右と左のどちらかの試料を運んで => REJECT (類似度=0.751, 差=0.018)
AI推論: 最初の姿勢へ戻って => HOME (類似度=0.739, 差=0.348)
AI推論: 左側の部品を検査台の前へ動かして => LEFT_SAMPLE (類似度=0.776, 差=0.107)
動画保存: ur5e_ai_demo_65s_v2.mp4  長さ=65 秒
結果ログ: ur5e_ai_demo_65s_v2_log.csv


     instruction      task_id  score  margin  max_joint_error_rad target_reached_within_0p12rad
左側の部品を検査台の前へ動かして  LEFT_SAMPLE 0.7765  0.1073               0.0129                          True
       最初の姿勢へ戻って         HOME 0.7386  0.3480               0.0081                          True
右にある試料を計測場所へ移動して RIGHT_SAMPLE 0.9876  0.0522               0.0129                          True
 右と左のどちらかの試料を運んで       REJECT 0.7515  0.0183               0.0131                           NaN
       最初の姿勢へ戻って         HOME 0.7386  0.3480               0.0081                          True
左側の部品を検査台の前へ動かして  LEFT_SAMPLE 0.7765  0.1073               0.0129                          True


In [24]:
# セル8: PCへダウンロード（全セルが成功してから）
from google.colab import files
files.download("ur5e_ai_demo_65s_v2.mp4")
files.download("ur5e_ai_demo_65s_v2_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 結果の見方・重要な限界

- 動画中の `AI task` が `LEFT_SAMPLE` / `RIGHT_SAMPLE` / `HOME` で変化し、UR5eの姿勢も変わっているか確認します。
- `max_joint_error_rad` は**実際の関節角と指定目標角の最大偏差**です。目標到達判定の0.12 radはこの教材の暫定的な確認値であり、安全基準ではありません。
- 実際の試料をつかんでいるわけではありません。`LEFT_SAMPLE` と `RIGHT_SAMPLE` は事前設定した関節角へ移動するタスクのラベルです。タスク名に「試料を運ぶ」とあっても、この段階の実装は運搬ではありません。
- AIはタスクIDを推論していますが、関節角・軌道は学習済みAIではなくプログラムが与えています。
- 未知の作業指示に対する認識率、衝突回避、実機上の安全性、把持能力は未評価です。
- 今回の成果をGitHubに登録する際は `.ipynb` とログCSVを登録し、MP4はYouTubeなどで共有します。個人情報や認証情報、コードセル出力の履歴も確認してください。

### 使用資料
- Google DeepMind, [MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie), UR5eモデル（BSD-3-Clause）
- [MuJoCo Python API](https://mujoco.readthedocs.io/en/stable/python.html)
- [Sentence Transformer 日本語を含む多言語モデル](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)